1. Design a Multi-layer perceptron to make a binary decision on the following datasets.

Use 70% of the dataset as training set and 30% as test set.

a. Adult Census Income dataset: https://www.kaggle.com/datasets/uciml/adult-
census-income/ - Predict whether the annual income of the person is >=50K or
<50K

b. Indian Liver Patient Records dataset:
https://www.kaggle.com/datasets/uciml/indian-liver-patient-records - Predict
whether person needs to be diagnosed or not?

c. Titanic: Machine Learning from Disaster dataset:
https://www.kaggle.com/c/titanic/data - Predict survival on the Titanic

In [42]:
import pandas as pd
import numpy as np

In [43]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "adult.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/adult-census-income",
  file_path
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

First 5 records:    age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356              

In [44]:
# print(np.random.randn(2, 3)*0.1)
np.zeros((3, 2))

array([[0., 0.],
       [0., 0.],
       [0., 0.]])

In [45]:
class MLP:
    def __init__(s, ip_size = 2, hidden_size = 4, op_size = 1, lr=0.1):
        s.lr = lr

        s.w1 = np.random.randn(ip_size, hidden_size) * 0.01
        s.b1 = np.zeroes((1, hidden_size))

        s.w2 = np.random.randn(hidden_size, op_size) * 0.01
        s.b1 = np.zeroes((1, op_size))

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    def tanh(self, z):
        return np.tanh(z)
    def sig_deriv(self, z):
        return z*(1-z)
    def tan_deriv(self, z):
        return 1 - z**2
    
    def forward_pass(self, X):
        self.z1 = np.dot(X, self.w1) + self.b1
        self.h1 = self.tanh(z1)

        self.z2 = np.dot(h1, self.w2) + self.b2
        self.h2 = self.sigmoid(z2)
        return h2

    def c_loss(self, y, y_pred):
        eps = 1e-15
        return -(np.sum(y*np.log(y_pred+eps) + (1-y)*np.log(1-y_pred+eps))) / y.shape[0]

    def back_pass(self, X, y):
        m = X.shape[0]

        dz2 = np.dot
        delta2 = (self.a2 - y.reshape(-1, 1)) * self.sigmoid_derivative(self.a2)
        dw2 = (self.a1.T @ delta2) / m
        db2 = np.mean(delta2, axis=0, keepdims=True)

        delta1 = (delta2 @ self.w2.T) * self.tanh_derivative(self.a1)
        dw1 = (X.T @ delta1) / m
        db1 = np.mean(delta1, axis=0, keepdims=True)

        self.w2 -= self.lr * dw2
        self.b2 -= self.lr * db2
        self.w1 -= self.lr * dw1
        self.b1 -= self.lr * db1

In [46]:
"""
Multi-Layer Perceptron (MLP) - Adult Census Income
====================================================
Goal: Predict if a person earns >50K or <=50K per year

WHAT IS AN MLP?
  - It's a neural network with layers of neurons
  - Input Layer  → takes in the features (age, education, etc.)
  - Hidden Layer → learns patterns from the data
  - Output Layer → gives a probability (0 to 1) → we threshold at 0.5

FLOW:
  Data → Preprocess → Train MLP → Evaluate on Test Set
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [47]:
# ──────────────────────────────────────────────────────────────
# STEP 1: LOAD DATA
# ──────────────────────────────────────────────────────────────

# If you have the actual Kaggle file, use:
# df = pd.read_csv("adult.csv")

# For now we generate data with the same structure as adult.csv
import kagglehub
from kagglehub import KaggleDatasetAdapter
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "uciml/adult-census-income",
    "adult.csv"
)

print("Shape:", df.shape)
print("\nFirst 3 rows:")
print(df.head(3))
print("\nTarget column distribution:")
print(df['income'].value_counts())

Shape: (32561, 15)

First 3 rows:
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   

        occupation   relationship   race     sex  capital.gain  capital.loss  \
0                ?  Not-in-family  White  Female             0          4356   
1  Exec-managerial  Not-in-family  White  Female             0          4356   
2                ?      Unmarried  Black  Female             0          4356   

   hours.per.week native.country income  
0              40  United-States  <=50K  
1              18  United-States  <=50K  
2              40  United-States  <=50K  

Target column distribution:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64


In [48]:
# ──────────────────────────────────────────────────────────────
# STEP 2: PREPROCESS DATA
# ──────────────────────────────────────────────────────────────

# --- 2a. Convert target to 0 or 1 ---
# ">50K" → 1,  "<=50K" → 0
df['income'] = df['income'].str.strip()          # remove spaces
df['income'] = (df['income'].str.contains('>50K')).astype(int)

# --- 2b. Fill missing values ---
# Some columns have "?" — replace with the most common value (mode)
df.replace('?', np.nan, inplace=True)
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())        # numeric → median
    else:
        df[col] = df[col].fillna(df[col].mode()[0])       # categorical → most common

# --- 2c. Encode categorical columns to numbers ---
# ML models only understand numbers, not strings like "Male" / "Female"
le = LabelEncoder()
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = le.fit_transform(df[col])

# --- 2d. Separate features (X) and target (y) ---
X = df.drop('income', axis=1).values   # everything except income
y = df['income'].values                 # just the income column

print(f"\nFeatures shape: {X.shape}")   # (rows, columns)
print(f"Target shape:   {y.shape}")

# --- 2e. Split: 70% train, 30% test ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"\nTraining samples: {len(X_train)}")
print(f"Testing  samples: {len(X_test)}")

# --- 2f. Scale features to have mean=0, std=1 ---
# This helps the network learn faster and more stably
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # learn scale from train, apply to train
X_test  = scaler.transform(X_test)        # apply same scale to test (don't re-learn!)


Features shape: (32561, 14)
Target shape:   (32561,)

Training samples: 22792
Testing  samples: 9769


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 3: BUILD THE MLP
# ──────────────────────────────────────────────────────────────

class MLP:
    """
    A simple Multi-Layer Perceptron with one hidden layer.

    Architecture:
      [Input] → [Hidden Layer (ReLU)] → [Output (Sigmoid)]

    For adult.csv with 14 features and 64 hidden neurons:
      14 → 64 → 1
    """

    def __init__(self, input_size, hidden_size, learning_rate=0.001):
        """
        Initialize weights randomly.
        We use small random values so the network doesn't saturate early.

        Weights:
          W1: connects Input → Hidden  (shape: input_size × hidden_size)
          W2: connects Hidden → Output (shape: hidden_size × 1)
        Biases allow the neuron to shift its output.
        """
        self.lr = learning_rate

        # He initialization: helps with ReLU activation
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, 1) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, 1))

        self.losses = []  # track loss each epoch

    # ── Activation Functions ─────────────────────────────────

    def relu(self, z):
        """
        ReLU (Rectified Linear Unit): max(0, z)
        - If z > 0 → keep it
        - If z < 0 → make it 0
        Used in hidden layers. Simple and effective.
        """
        return np.maximum(0, z)

    def relu_derivative(self, z):
        """
        Derivative of ReLU:
        - 1 where z > 0
        - 0 where z <= 0
        Used during backpropagation.
        """
        return (z > 0).astype(float)

    def sigmoid(self, z):
        """
        Sigmoid: 1 / (1 + e^(-z))
        Squashes any value to range (0, 1) → acts as a probability.
        Used in the output layer for binary classification.
        """
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    # ── Loss Function ────────────────────────────────────────

    def binary_cross_entropy(self, y_true, y_pred):
        """
        Measures how wrong our predictions are.

        Formula: -mean( y*log(p) + (1-y)*log(1-p) )
        - If y=1 and p≈1 → loss ≈ 0  (correct!)
        - If y=1 and p≈0 → loss is large (wrong!)
        """
        eps = 1e-15
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    # ── Forward Pass ─────────────────────────────────────────

    def forward(self, X):
        """
        Pass input through each layer to get a prediction.

        Layer 1 (Hidden):
          z1 = X @ W1 + b1   ← linear combination
          a1 = ReLU(z1)       ← apply activation

        Layer 2 (Output):
          z2 = a1 @ W2 + b2
          a2 = Sigmoid(z2)    ← probability between 0 and 1

        We save z1 and a1 because we need them for backpropagation.
        """
        self.z1 = X @ self.W1 + self.b1   # hidden layer (pre-activation)
        self.a1 = self.relu(self.z1)       # hidden layer (post-activation)

        self.z2 = self.a1 @ self.W2 + self.b2  # output layer (pre-activation)
        self.a2 = self.sigmoid(self.z2)         # output = probability

        return self.a2

    # ── Backward Pass (Backpropagation) ──────────────────────

    def backward(self, X, y_true):
        """
        Compute how much each weight contributed to the error,
        then adjust weights to reduce the error.

        This uses the chain rule from calculus:
          How does the loss change when we tweak W2?
          → dL/dW2 = a1.T @ dL/dz2

          How does the loss change when we tweak W1?
          → dL/dW1 = X.T @ (dL/dz2 @ W2.T * relu'(z1))

        Steps:
          1. Compute error at output (delta2)
          2. Propagate error back to hidden layer (delta1)
          3. Compute gradients for each weight matrix
          4. Update weights: W = W - lr * gradient
        """
        m = X.shape[0]  # number of samples in this batch

        # --- Output layer gradient ---
        # dL/dz2 = (prediction - true label)  [combined BCE + sigmoid derivative]
        delta2 = self.a2 - y_true.reshape(-1, 1)

        # Gradient for W2 and b2
        dW2 = (self.a1.T @ delta2) / m
        db2 = np.mean(delta2, axis=0, keepdims=True)

        # --- Hidden layer gradient ---
        # Propagate error backwards through W2, then through ReLU
        delta1 = (delta2 @ self.W2.T) * self.relu_derivative(self.z1)

        # Gradient for W1 and b1
        dW1 = (X.T @ delta1) / m
        db1 = np.mean(delta1, axis=0, keepdims=True)

        # --- Update weights (Gradient Descent) ---
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    # ── Training Loop ────────────────────────────────────────

    def train(self, X, y, epochs=100, batch_size=64):
        """
        Train the network for multiple epochs.

        Each epoch:
          1. Shuffle the data
          2. Split into mini-batches
          3. For each batch: forward → compute loss → backward → update
        """
        n = X.shape[0]

        for epoch in range(1, epochs + 1):
            # Shuffle data so we don't learn in the same order every time
            idx = np.random.permutation(n)
            X, y = X[idx], y[idx]

            epoch_loss = 0
            num_batches = 0

            for start in range(0, n, batch_size):
                X_batch = X[start : start + batch_size]
                y_batch = y[start : start + batch_size]

                # Forward: get predictions
                y_pred = self.forward(X_batch)

                # Loss: how wrong are we?
                loss = self.binary_cross_entropy(y_batch, y_pred)
                epoch_loss += loss
                num_batches += 1

                # Backward: update weights
                self.backward(X_batch, y_batch)

            avg_loss = epoch_loss / num_batches
            self.losses.append(avg_loss)

            if epoch % 10 == 0:
                print(f"  Epoch {epoch:3d}/{epochs}  |  Loss: {avg_loss:.4f}")

    # ── Prediction ───────────────────────────────────────────

    def predict(self, X, threshold=0.5):
        """
        Get binary predictions (0 or 1).
        If probability >= 0.5 → predict 1 (>50K)
        If probability <  0.5 → predict 0 (<=50K)
        """
        probs = self.forward(X)
        return (probs >= threshold).astype(int).flatten()


# ──────────────────────────────────────────────────────────────
# STEP 4: TRAIN THE MODEL
# ──────────────────────────────────────────────────────────────

print("\n" + "="*50)
print("  Training the MLP...")
print("="*50)

# Create the MLP
# 14 input features → 64 hidden neurons → 1 output
model = MLP(
    input_size=X_train.shape[1],   # number of features (14)
    hidden_size=64,                 # number of neurons in hidden layer
    learning_rate=0.001             # how big each update step is
)

model.train(X_train, y_train, epochs=100, batch_size=64)


# ──────────────────────────────────────────────────────────────
# STEP 5: EVALUATE ON TEST SET
# ──────────────────────────────────────────────────────────────

print("\n" + "="*50)
print("  Evaluating on Test Set...")
print("="*50)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n  Test Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# ──────────────────────────────────────────────────────────────
# STEP 6: PLOT RESULTS
# ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('MLP on Adult Census Income Dataset', fontsize=13, fontweight='bold')

# --- Plot 1: Training Loss Curve ---
axes[0].plot(model.losses, color='steelblue', lw=2)
axes[0].set_title('Training Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
im = axes[1].imshow(cm, cmap='Blues')
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Pred <=50K', 'Pred >50K'])
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(['True <=50K', 'True >50K'])
axes[1].set_title(f'Confusion Matrix  (Accuracy: {acc*100:.1f}%)')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                     fontsize=14, color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.savefig('mlp_adult_results.png', dpi=130, bbox_inches='tight')
print("\n  Plot saved as mlp_adult_results.png")
print("\nDone!")


  Training the MLP...
  Epoch  10/100  |  Loss: 0.6753
  Epoch  20/100  |  Loss: 0.7294
  Epoch  30/100  |  Loss: 0.7659
  Epoch  40/100  |  Loss: 0.7934
  Epoch  50/100  |  Loss: 0.8114
  Epoch  60/100  |  Loss: 0.8306
  Epoch  70/100  |  Loss: 0.8436
  Epoch  80/100  |  Loss: 0.8550
  Epoch  90/100  |  Loss: 0.8664
  Epoch 100/100  |  Loss: 0.8782

  Evaluating on Test Set...

  Test Accuracy: 83.49%

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.86      0.94      0.90      7417
        >50K       0.72      0.52      0.60      2352

    accuracy                           0.83      9769
   macro avg       0.79      0.73      0.75      9769
weighted avg       0.83      0.83      0.83      9769

Confusion Matrix:
[[6937  480]
 [1133 1219]]

  Plot saved as mlp_adult_results.png

Done!
